# SHIELD regime analysis — is permeation diffusion- or surface-limited?

Steady-state permeation of hydrogen through a metal sample is a chain of steps:
dissociative adsorption at the upstream surface, diffusion through the bulk, and
recombinative desorption downstream. Whichever step is slowest sets the *regime*,
and the regime decides which material properties a run can give you:

| regime | steady flux | what a run yields |
|---|---|---|
| **diffusion-limited (DL)** | J = Φ·√P / e | permeability Φ, diffusivity D (time lag), solubility S |
| **surface-limited (SL)** | J = K_d·P / 2 | dissociation coefficient K_d (and K_r via detailed balance) |

The two laws differ in their pressure exponent — J ∝ P^0.5 (Sieverts scaling)
vs J ∝ P — so the regime is **measured, not assumed**: repeat the run at one
temperature over a range of upstream pressures and fit the log-log slope of
flux vs pressure (`fit_pressure_exponent`). A slope in between means *mixed*
transport, where neither limiting formula applies.

This notebook runs that workflow on the bare carbon-steel campaign of
January–February 2026: two temperature groups (~550 K and ~493 K) with
upstream pressures spanning 24–573 Torr.

Prerequisite: the environment from the README's *Local development setup*
(`uv sync`, plus `uv pip install -e ../SHIELD-Data` for `shield_data`).

In [ ]:
%matplotlib inline
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from shield_toolbox import fetch_run, load_results, process_run
from shield_toolbox.analysis import fit_pressure_exponent
from shield_toolbox.constants import TORR_TO_PA

## 1. Process the pressure-sweep campaign

Thirteen runs on the same uncoated carbon-steel sample (0.65 mm), differing
only in upstream pressure and furnace setpoint. Each is processed exactly like
any other run — the regime work happens on the aggregated results.

In [ ]:
RUN_IDS = [
    # ~550 K (furnace setpoint 300 °C), 24-573 Torr
    "26.01.18_run_1_14h05",
    "26.01.18_run_2_20h10",
    "26.01.20_run_1_15h19",
    "26.01.21_run_1_09h49",
    "26.01.21_run_2_14h01",
    "26.01.23_run_1_09h57",
    "26.01.27_run_1_16h43",
    "26.01.28_run_1_10h52",
    "26.02.03_run_1_14h56",
    # ~493 K (furnace setpoint 250 °C), 52-374 Torr
    "26.02.12_run_1_17h13",
    "26.02.15_run_1_16h26",
    "26.02.17_run_1_15h30",
    "26.02.18_run_1_10h50",
]

output_dir = Path(tempfile.mkdtemp())
for run_id in RUN_IDS:
    process_run(fetch_run(run_id)).write(output_dir)

results = load_results(output_dir, substrate="carbon steel", coating="none")
print(f"{len(results)} runs processed")

## 2. Recover the steady-state flux

`process_run` reports the *apparent* permeability Φ = J·e/√P — the
diffusion-limited interpretation of the measured flux. Inverting that gives
back the model-free measured flux, J = Φ·√P/e, which is what the regime fit
works on.

In [ ]:
p_up_pa = results.upstream_torr * TORR_TO_PA
results["flux"] = results.permeability * np.sqrt(p_up_pa) / results.thickness_m
results["flux_err"] = results.permeability_err * np.sqrt(p_up_pa) / results.thickness_m

results[["run_id", "temperature_K", "upstream_torr", "flux", "flux_err"]]

## 3. Group by temperature and fit the pressure exponent

The exponent only means something within one temperature group. At 550 K the
sweep straddles the regime crossover (~50 Torr for this sample), so the low-
and high-pressure sides are fitted separately.

In [ ]:
group_550 = results[(results.temperature_K - 550).abs() < 20]
group_493 = results[(results.temperature_K - 493).abs() < 20]

low_550 = group_550[group_550.upstream_torr < 50]
high_550 = group_550[group_550.upstream_torr > 50]

n_low = fit_pressure_exponent(low_550.upstream_torr, low_550.flux)
n_high = fit_pressure_exponent(high_550.upstream_torr, high_550.flux)
n_493 = fit_pressure_exponent(group_493.upstream_torr, group_493.flux)

print("guide: n = 0.5 → diffusion-limited, n = 1 → surface-limited")
print(f"550 K, below 50 Torr : n = {n_low}")
print(f"550 K, above 50 Torr : n = {n_high}")
print(f"493 K, all pressures : n = {n_493}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
for group, label, color in [(group_550, "~550 K", "C0"), (group_493, "~493 K", "C1")]:
    ax.errorbar(
        group.upstream_torr,
        group.flux,
        yerr=group.flux_err,
        fmt="o",
        color=color,
        capsize=3,
        label=label,
    )


def draw_power_law(subset, exponent, color):
    p = np.logspace(
        np.log10(subset.upstream_torr.min()), np.log10(subset.upstream_torr.max()), 50
    )
    scale = np.median(subset.flux / subset.upstream_torr**exponent.n)
    ax.plot(
        p,
        scale * p**exponent.n,
        "--",
        color=color,
        label=f"fit: J ∝ P^{exponent.n:.2f}",
    )


draw_power_law(low_550, n_low, "C3")
draw_power_law(high_550, n_high, "C2")
ax.axvline(50, color="gray", ls=":", lw=1)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Upstream pressure (Torr)")
ax.set_ylabel("Steady-state flux (H/(m²·s))")
ax.legend()
fig.tight_layout()

## 4. Verdict, and where to go next

- **550 K, above ~50 Torr — diffusion-limited** (n ≈ 0.5): these runs support
  the full Φ / D / S extraction —
  [`diffusion_limited_analysis.ipynb`](diffusion_limited_analysis.ipynb).
- **550 K, below ~50 Torr — surface-limited** (n ≈ 1): the DL quantities are
  meaningless here; extract the surface coefficients instead —
  [`surface_limited_analysis.ipynb`](surface_limited_analysis.ipynb).
- **493 K — predominantly diffusion-limited, with the onset of surface
  influence** (n ≈ 0.58 ± 0.04, measurably above ½): this sweep only spans
  52–374 Torr, so it never reaches the surface-limited side; points below
  ~50 Torr would be needed to pin the crossover at this temperature. Treat DL
  extractions from these runs with care, and extract nothing with the
  surface-limited shortcut — a mixed slope means neither limiting formula is
  exact.

A new sample or temperature needs its own sweep — the crossover pressure
moves with temperature and with the surface condition (oxide state, roughness,
coating).